# 📊 Data Preparation for Fine-Tuning

**Prepare high-quality training data for LLM fine-tuning**

---

## 📋 Overview

**What you'll learn:**
- Data format requirements
- Quality vs quantity tradeoffs
- Data cleaning and validation
- Creating training examples
- Train/validation splits

**Time estimate:** ⏱️ 60 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
import json
import pandas as pd
from typing import List, Dict
from collections import Counter
import re

print("✅ Setup complete")

## 🤔 Why Data Quality Matters

### The Truth About Fine-Tuning Data:

```
❌ Common Myth:
"More data = better model"

✅ Reality:
"High-quality, diverse data = better model"
```

### Data Quality Impact:

| Dataset | Size | Quality | Model Performance |
|---------|------|---------|-------------------|
| Dataset A | 10,000 | Low | 65% accuracy |
| Dataset B | 1,000 | High | 85% accuracy |
| Dataset C | 500 | Excellent | 90% accuracy |

### Key Principles:

1. **Quality > Quantity**
   - 100 perfect examples > 10,000 noisy ones
   - Focus on representative, diverse data

2. **Format Consistency**
   - Same structure across all examples
   - Consistent prompt templates

3. **Diversity**
   - Cover all use cases
   - Include edge cases
   - Vary input styles

## 📝 Data Format Requirements

### OpenAI Fine-Tuning Format (JSONL):

In [ ]:
# Standard format: Chat completion
openai_format_example = {
    "messages": [
        {"role": "system", "content": "You are a helpful customer support agent."},
        {"role": "user", "content": "How do I reset my password?"},
        {"role": "assistant", "content": "To reset your password: 1. Go to login page 2. Click 'Forgot Password' 3. Enter your email 4. Check your inbox for reset link"}
    ]
}

print("📄 OpenAI Fine-Tuning Format:")
print(json.dumps(openai_format_example, indent=2))

# Multi-turn conversation
multi_turn_example = {
    "messages": [
        {"role": "system", "content": "You are a Python programming tutor."},
        {"role": "user", "content": "What is a list in Python?"},
        {"role": "assistant", "content": "A list is an ordered collection of items, created using square brackets: [1, 2, 3]"},
        {"role": "user", "content": "How do I add items?"},
        {"role": "assistant", "content": "Use the .append() method: my_list.append(4)"}
    ]
}

print("\n📄 Multi-Turn Example:")
print(json.dumps(multi_turn_example, indent=2))

## 🏗️ Creating Training Examples

In [ ]:
class DatasetBuilder:
    """Build fine-tuning datasets."""
    
    def __init__(self, system_prompt: str = None):
        self.system_prompt = system_prompt
        self.examples = []
    
    def add_example(
        self,
        user_message: str,
        assistant_message: str,
        context_messages: List[Dict] = None
    ):
        """Add a training example."""
        
        messages = []
        
        # Add system prompt if provided
        if self.system_prompt:
            messages.append({
                "role": "system",
                "content": self.system_prompt
            })
        
        # Add context (for multi-turn)
        if context_messages:
            messages.extend(context_messages)
        
        # Add user and assistant messages
        messages.append({"role": "user", "content": user_message})
        messages.append({"role": "assistant", "content": assistant_message})
        
        self.examples.append({"messages": messages})
    
    def save_jsonl(self, filename: str):
        """Save dataset in JSONL format."""
        with open(filename, 'w') as f:
            for example in self.examples:
                f.write(json.dumps(example) + '\n')
        
        print(f"✅ Saved {len(self.examples)} examples to {filename}")
    
    def get_stats(self) -> Dict:
        """Get dataset statistics."""
        total_examples = len(self.examples)
        
        user_lengths = []
        assistant_lengths = []
        
        for example in self.examples:
            for msg in example['messages']:
                if msg['role'] == 'user':
                    user_lengths.append(len(msg['content']))
                elif msg['role'] == 'assistant':
                    assistant_lengths.append(len(msg['content']))
        
        return {
            'total_examples': total_examples,
            'avg_user_length': sum(user_lengths) / len(user_lengths) if user_lengths else 0,
            'avg_assistant_length': sum(assistant_lengths) / len(assistant_lengths) if assistant_lengths else 0,
        }

# Create a customer support dataset
builder = DatasetBuilder(system_prompt="You are a helpful customer support agent for TechCo.")

# Add examples
builder.add_example(
    user_message="How do I track my order?",
    assistant_message="You can track your order by: 1. Logging into your account 2. Going to 'My Orders' 3. Clicking on the order number. You'll see real-time tracking information."
)

builder.add_example(
    user_message="What is your return policy?",
    assistant_message="Our return policy allows returns within 30 days of purchase. Items must be unused and in original packaging. To start a return, visit the Returns page in your account."
)

builder.add_example(
    user_message="I forgot my password",
    assistant_message="No problem! Click 'Forgot Password' on the login page, enter your email, and we'll send you a reset link within a few minutes. Check your spam folder if you don't see it."
)

print("📊 Dataset Statistics:")
stats = builder.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

# Save dataset
builder.save_jsonl('/home/user/Rag_notebooks/06_fine_tuning/sample_dataset.jsonl')

## 🧹 Data Cleaning and Validation

In [ ]:
class DataValidator:
    """Validate fine-tuning datasets."""
    
    @staticmethod
    def load_jsonl(filename: str) -> List[Dict]:
        """Load JSONL file."""
        examples = []
        with open(filename, 'r') as f:
            for line in f:
                examples.append(json.loads(line))
        return examples
    
    @staticmethod
    def validate_format(examples: List[Dict]) -> Dict:
        """Check format validity."""
        issues = []
        
        for i, example in enumerate(examples):
            # Check required fields
            if 'messages' not in example:
                issues.append(f"Example {i}: Missing 'messages' field")
                continue
            
            messages = example['messages']
            
            # Check message structure
            for j, msg in enumerate(messages):
                if 'role' not in msg:
                    issues.append(f"Example {i}, message {j}: Missing 'role'")
                if 'content' not in msg:
                    issues.append(f"Example {i}, message {j}: Missing 'content'")
                
                # Check valid roles
                if msg.get('role') not in ['system', 'user', 'assistant']:
                    issues.append(f"Example {i}, message {j}: Invalid role '{msg.get('role')}'")
            
            # Check ends with assistant message
            if messages and messages[-1].get('role') != 'assistant':
                issues.append(f"Example {i}: Must end with assistant message")
        
        return {
            'valid': len(issues) == 0,
            'issues': issues,
            'num_examples': len(examples)
        }
    
    @staticmethod
    def check_quality(examples: List[Dict]) -> Dict:
        """Check data quality metrics."""
        
        # Length statistics
        assistant_lengths = []
        user_lengths = []
        
        for example in examples:
            for msg in example['messages']:
                content_len = len(msg['content'])
                if msg['role'] == 'assistant':
                    assistant_lengths.append(content_len)
                elif msg['role'] == 'user':
                    user_lengths.append(content_len)
        
        # Check for duplicates
        assistant_messages = []
        for example in examples:
            for msg in example['messages']:
                if msg['role'] == 'assistant':
                    assistant_messages.append(msg['content'])
        
        duplicates = len(assistant_messages) - len(set(assistant_messages))
        
        return {
            'avg_user_length': sum(user_lengths) / len(user_lengths) if user_lengths else 0,
            'avg_assistant_length': sum(assistant_lengths) / len(assistant_lengths) if assistant_lengths else 0,
            'max_assistant_length': max(assistant_lengths) if assistant_lengths else 0,
            'min_assistant_length': min(assistant_lengths) if assistant_lengths else 0,
            'duplicate_responses': duplicates,
        }

# Validate our dataset
validator = DataValidator()
examples = validator.load_jsonl('/home/user/Rag_notebooks/06_fine_tuning/sample_dataset.jsonl')

print("🔍 Format Validation:")
format_check = validator.validate_format(examples)
if format_check['valid']:
    print("  ✅ All examples are properly formatted")
else:
    print(f"  ❌ Found {len(format_check['issues'])} issues:")
    for issue in format_check['issues'][:5]:  # Show first 5
        print(f"    - {issue}")

print("\n📊 Quality Metrics:")
quality = validator.check_quality(examples)
for key, value in quality.items():
    print(f"  {key}: {value:.1f}" if isinstance(value, float) else f"  {key}: {value}")

## 📈 Data Augmentation

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

class DataAugmenter:
    """Augment training data using LLM."""
    
    @staticmethod
    def paraphrase_user_message(message: str, num_variations: int = 2) -> List[str]:
        """Generate paraphrases of user messages."""
        
        prompt = f"""Generate {num_variations} different ways a user might ask the following question.
Keep the same meaning but use different words and phrasing.

Original: {message}

Variations (one per line):"""
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8
        )
        
        variations_text = response.choices[0].message.content.strip()
        variations = [line.strip() for line in variations_text.split('\n') if line.strip()]
        
        # Clean up numbering
        cleaned = []
        for var in variations:
            # Remove leading numbers like "1. " or "1) "
            cleaned_var = re.sub(r'^\d+[.)]\s*', '', var)
            if cleaned_var:
                cleaned.append(cleaned_var)
        
        return cleaned[:num_variations]
    
    @staticmethod
    def augment_dataset(
        examples: List[Dict],
        augmentation_factor: int = 2
    ) -> List[Dict]:
        """Augment dataset by creating variations."""
        
        augmented = list(examples)  # Start with original
        
        for example in examples:
            messages = example['messages']
            
            # Find user message
            user_msg_idx = None
            for i, msg in enumerate(messages):
                if msg['role'] == 'user':
                    user_msg_idx = i
                    break
            
            if user_msg_idx is None:
                continue
            
            user_message = messages[user_msg_idx]['content']
            
            # Generate variations
            variations = DataAugmenter.paraphrase_user_message(
                user_message,
                num_variations=augmentation_factor - 1
            )
            
            # Create new examples with variations
            for variation in variations:
                new_messages = messages.copy()
                new_messages[user_msg_idx] = {
                    "role": "user",
                    "content": variation
                }
                augmented.append({"messages": new_messages})
        
        return augmented

# Test augmentation
print("🔄 Data Augmentation Example\n")

original_message = "How do I reset my password?"
print(f"Original: {original_message}\n")

print("Generated variations:")
variations = DataAugmenter.paraphrase_user_message(original_message, num_variations=3)
for i, var in enumerate(variations, 1):
    print(f"  {i}. {var}")

print("\n💡 Use augmentation to increase dataset size while maintaining quality")

## 🎯 Train/Validation Split

In [ ]:
import random
from typing import Tuple

def train_val_split(
    examples: List[Dict],
    val_ratio: float = 0.2,
    shuffle: bool = True
) -> Tuple[List[Dict], List[Dict]]:
    """Split dataset into train and validation sets."""
    
    if shuffle:
        examples = examples.copy()
        random.shuffle(examples)
    
    split_idx = int(len(examples) * (1 - val_ratio))
    
    train = examples[:split_idx]
    val = examples[split_idx:]
    
    return train, val

def save_split_datasets(
    train: List[Dict],
    val: List[Dict],
    train_file: str,
    val_file: str
):
    """Save train and validation datasets."""
    
    with open(train_file, 'w') as f:
        for example in train:
            f.write(json.dumps(example) + '\n')
    
    with open(val_file, 'w') as f:
        for example in val:
            f.write(json.dumps(example) + '\n')
    
    print(f"✅ Saved {len(train)} training examples to {train_file}")
    print(f"✅ Saved {len(val)} validation examples to {val_file}")

# Create train/val split
examples = validator.load_jsonl('/home/user/Rag_notebooks/06_fine_tuning/sample_dataset.jsonl')

train, val = train_val_split(examples, val_ratio=0.2)

print(f"📊 Dataset Split:")
print(f"  Total: {len(examples)}")
print(f"  Training: {len(train)} ({len(train)/len(examples)*100:.1f}%)")
print(f"  Validation: {len(val)} ({len(val)/len(examples)*100:.1f}%)")

# Save splits
save_split_datasets(
    train,
    val,
    '/home/user/Rag_notebooks/06_fine_tuning/train.jsonl',
    '/home/user/Rag_notebooks/06_fine_tuning/val.jsonl'
)

## ✅ Summary

### Data Preparation Checklist:

**1. Format Requirements:**
```jsonl
// Each line is a JSON object
{"messages": [
  {"role": "system", "content": "..."},
  {"role": "user", "content": "..."},
  {"role": "assistant", "content": "..."}
]}
```

**2. Quality Guidelines:**
- ✅ **500-1,000 examples** for good results
- ✅ **Diverse** examples covering all use cases
- ✅ **Consistent** formatting
- ✅ **High-quality** responses (no typos, accurate)
- ✅ **Representative** of production queries

**3. Validation Steps:**
```python
# Check format
validator.validate_format(examples)

# Check quality
validator.check_quality(examples)

# Check for duplicates
# Check length distribution
# Verify diversity
```

**4. Data Splits:**
```
Training:   80% (for learning)
Validation: 20% (for evaluation)
```

### How Much Data Do You Need?

| Task Complexity | Minimum | Recommended | Ideal |
|-----------------|---------|-------------|-------|
| **Simple** (Style, tone) | 50 | 200 | 500+ |
| **Medium** (Customer support) | 200 | 500 | 1,000+ |
| **Complex** (Technical QA) | 500 | 1,000 | 3,000+ |

### Common Data Issues:

❌ **Bad:**
```jsonl
// Missing system prompt
// Inconsistent response style
// Duplicate examples
// Too short responses
```

✅ **Good:**
```jsonl
// Clear system prompt
// Consistent, professional tone
// Diverse examples
// Complete, helpful responses
```

### Data Sources:

1. **Historical data**
   - Customer support tickets
   - Chat logs
   - Email responses

2. **Synthetic data**
   - LLM-generated examples
   - Augmented variations
   - Template-based generation

3. **Human-created**
   - Subject matter experts
   - Crowdsourcing
   - Internal team

### Best Practices:

1. **Start small, iterate**
   - Begin with 100-200 examples
   - Fine-tune and evaluate
   - Add more data where model struggles

2. **Focus on edge cases**
   - Include difficult examples
   - Cover unusual queries
   - Add examples where base model fails

3. **Maintain quality**
   - Review all examples
   - Remove duplicates
   - Fix typos and errors

4. **Version your data**
   ```python
   # Keep track of dataset versions
   dataset_v1.jsonl
   dataset_v2.jsonl  # Added 200 examples
   dataset_v3.jsonl  # Fixed quality issues
   ```

### Production Tips:

```python
# 1. Validate before uploading
validator.validate_format(examples)

# 2. Check token counts (cost estimation)
total_tokens = sum(count_tokens(ex) for ex in examples)

# 3. Monitor data drift
# Compare production queries to training data

# 4. Continuous improvement
# Add production examples that failed
# Retrain periodically
```

### Next: `06_fine_tuning/03_openai_finetuning.ipynb`